# Load Final Dataset 

In [6]:
import pandas as pd

df = pd.read_csv(
    "../datasets/processed/climate_heatwave_predictions.csv"
)

print("Dataset Loaded Successfully")
print(df.shape)

print(df.columns.tolist())


Dataset Loaded Successfully
(119398, 150)
['latitude', 'longitude', 'temperature_celsius', 'wind_kph', 'wind_degree', 'pressure_mb', 'precip_mm', 'humidity', 'cloud', 'feels_like_celsius', 'visibility_km', 'uv_index', 'gust_kph', 'air_quality_Carbon_Monoxide', 'air_quality_Ozone', 'air_quality_Nitrogen_dioxide', 'air_quality_Sulphur_dioxide', 'air_quality_PM2.5', 'air_quality_PM10', 'air_quality_us-epa-index', 'air_quality_gb-defra-index', 'moon_illumination', 'year', 'month', 'day', 'hour', 'weekday', 'day_length_minutes', 'temperature_gap', 'pm_difference', 'pollution_intensity', 'wind_humidity_interaction', 'humidity_cloud_interaction', 'heatwave_index', 'region_1', 'region_2', 'region_3', 'region_4', 'region_5', 'region_6', 'region_7', 'region_8', 'region_9', 'region_10', 'region_11', 'region_12', 'region_13', 'region_14', 'region_15', 'region_16', 'region_17', 'region_18', 'region_19', 'region_20', 'region_21', 'region_22', 'region_23', 'region_24', 'region_25', 'region_26', 'regi

# Verify Rainfall Risk Exists


In [7]:

print(
    "rainfall_risk" in df.columns
)
if "rainfall_risk" in df.columns:
    print(
        df["rainfall_risk"]
        .value_counts()
    )


True
rainfall_risk
Low       104510
Medium     12184
High        2704
Name: count, dtype: int64


# Load Final Phase Outputs

In [8]:

import pandas as pd
profiles_df = pd.read_csv(
    "../datasets/processed/climate_profiles.csv"
)
anomaly_df = pd.read_csv(
    "../datasets/processed/climate_anomalies.csv"
)
rainfall_df = pd.read_csv(
    "../datasets/processed/climate_rainfall_predictions.csv"
)
heatwave_df = pd.read_csv(
    "../datasets/processed/climate_heatwave_predictions.csv"
)
print(profiles_df.shape)
print(anomaly_df.shape)
print(rainfall_df.shape)
print(heatwave_df.shape)


(119398, 146)
(119398, 148)
(119398, 149)
(119398, 150)


# CREATE MASTER DATASET

In [9]:

master_df = heatwave_df.copy()
master_df["rainfall_risk"] = rainfall_df["rainfall_risk"]
master_df["climate_profile"] = profiles_df["climate_profile"]
master_df["anomaly_status"] = anomaly_df["anomaly_status"]
print(master_df.shape)
print(
    master_df[
        [
            "climate_profile",
            "anomaly_status",
            "rainfall_risk",
            "heatwave_risk"
        ]
    ].head()
)


(119398, 150)
  climate_profile anomaly_status rainfall_risk heatwave_risk
0        Moderate         Normal           Low       Warning
1        Moderate         Normal           Low       Warning
2        Moderate         Normal           Low       Warning
3        Moderate         Normal           Low       Warning
4        Moderate         Normal           Low       Warning


# RISK SCORE MAPPINGS

In [10]:

rainfall_scores = {
    "Low": 10,
    "Medium": 40,
    "High": 80
}
heatwave_scores = {
    "Safe": 10,
    "Warning": 50,
    "Critical": 90
}
anomaly_scores = {
    "Normal": 0,
    "Anomaly": 100
}
profile_scores = {
    "Moderate": 20,
    "Pollution-Prone": 50,
    "Flood-Prone": 70,
    "Extreme-Pollution": 90
}


# CLIMATE RISK SCORE

In [11]:

master_df["rainfall_score"] = (
    master_df["rainfall_risk"]
    .map(rainfall_scores)
)
master_df["heatwave_score"] = (
    master_df["heatwave_risk"]
    .map(heatwave_scores)
)
master_df["anomaly_score_final"] = (
    master_df["anomaly_status"]
    .map(anomaly_scores)
)

master_df["profile_score"] = (
    master_df["climate_profile"]
    .map(profile_scores)
)
master_df["climate_risk_score"] = (
      0.30 * master_df["rainfall_score"]
    + 0.30 * master_df["heatwave_score"]
    + 0.20 * master_df["profile_score"]
    + 0.20 * master_df["anomaly_score_final"]
)
master_df["climate_risk_score"] = (
    master_df["climate_risk_score"]
    .round(0)
)
print(
    master_df["climate_risk_score"]
    .describe()
)


count    119398.000000
mean         18.155497
std           9.993843
min          10.000000
25%          10.000000
50%          16.000000
75%          22.000000
max          85.000000
Name: climate_risk_score, dtype: float64


# RISK CATEGORY

In [12]:

def risk_category(score):
    if score < 30:
        return "Low"
    elif score < 60:
        return "Medium"
    elif score < 80:
        return "High"
    else:
        return "Critical"

master_df["climate_risk"] = (
    master_df["climate_risk_score"]
    .apply(risk_category)
)
print(
    master_df["climate_risk"]
    .value_counts()
)
print("\nPercentage Distribution")
print(
    round(
        master_df["climate_risk"]
        .value_counts(normalize=True) * 100,
        2
    )
)


climate_risk
Low         100815
Medium       18210
High           359
Critical        14
Name: count, dtype: int64

Percentage Distribution
climate_risk
Low         84.44
Medium      15.25
High         0.30
Critical     0.01
Name: proportion, dtype: float64


# SAVE FINAL DATASET


In [13]:
master_df.to_csv(
    "../datasets/processed/climate_risk_intelligence.csv",
    index=False
)
print(
    "Climate Risk Intelligence Dataset Saved Successfully"
)
print(master_df.shape)


Climate Risk Intelligence Dataset Saved Successfully
(119398, 156)


# CLIMATE RISK SCORE

In [14]:

master_df["rainfall_score"] = (
    master_df["rainfall_risk"]
    .map(rainfall_scores)
)

master_df["heatwave_score"] = (
    master_df["heatwave_risk"]
    .map(heatwave_scores)
)

master_df["anomaly_score_final"] = (
    master_df["anomaly_status"]
    .map(anomaly_scores)
)

master_df["profile_score"] = (
    master_df["climate_profile"]
    .map(profile_scores)
)

master_df["climate_risk_score"] = (
      0.30 * master_df["rainfall_score"]
    + 0.30 * master_df["heatwave_score"]
    + 0.20 * master_df["profile_score"]
    + 0.20 * master_df["anomaly_score_final"]
)

master_df["climate_risk_score"] = (
    master_df["climate_risk_score"]
    .round(0)
)

print(
    master_df["climate_risk_score"]
    .describe()
)


count    119398.000000
mean         18.155497
std           9.993843
min          10.000000
25%          10.000000
50%          16.000000
75%          22.000000
max          85.000000
Name: climate_risk_score, dtype: float64


# RISK CATEGORY

In [15]:

def risk_category(score):
    if score < 30:
        return "Low"
    elif score < 60:
        return "Medium"
    elif score < 80:
        return "High"
    else:
        return "Critical"

master_df["climate_risk"] = (
    master_df["climate_risk_score"]
    .apply(risk_category)
)
print(
    master_df["climate_risk"]
    .value_counts()
)
print("\nPercentage Distribution")
print(
    round(
        master_df["climate_risk"]
        .value_counts(normalize=True) * 100,
        2
    )
)


climate_risk
Low         100815
Medium       18210
High           359
Critical        14
Name: count, dtype: int64

Percentage Distribution
climate_risk
Low         84.44
Medium      15.25
High         0.30
Critical     0.01
Name: proportion, dtype: float64


# SAVE FINAL DATASET

In [16]:

master_df.to_csv(
    "../datasets/processed/climate_risk_intelligence.csv",
    index=False
)
print(
    "Climate Risk Intelligence Dataset Saved Successfully"
)
print(master_df.shape)


Climate Risk Intelligence Dataset Saved Successfully
(119398, 156)


# Recommended Final Risk Categories


In [17]:

def climate_risk_category(score):
    if score <= 16:
        return "Low"
    elif score <= 32:
        return "Medium"
    elif score <= 53:
        return "High"
    else:
        return "Critical"


In [18]:
master_df["climate_risk"] = (
    master_df["climate_risk_score"]
    .apply(climate_risk_category)
)